# Reliability and Agreement Metrics

This notebook computes project metrics from final codebooks.

Data sources:
- Human final master codebooks
- Human intercoder reliability codebooks
- LLM coding workbooks

Rules:
- Open-text fields are excluded from scoring.
- `Other` follow-up text fields are excluded.
- Multi-select fields use set logic for set-overlap metrics.

This notebook is the single source for research-grade scoring: full coding can be captured upstream, then projected to final Master/IRR schema for metric comparability.

In [ ]:
from pathlib import Path
import itertools
import re
import numpy as np
import pandas as pd

try:
    import krippendorff
except Exception as e:
    raise ImportError('Install dependency first: pip install krippendorff') from e

ROOT = Path('/Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project')
HUMAN = ROOT / 'Codebooks' / 'Master Codebooks - Human'
LLM = ROOT / 'Codebooks' / 'LLM Codebook'
OUT = ROOT / 'Notebooks' / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

PATHS = {
    'master_content': HUMAN / 'Master Content Codebook - Final.xlsx',
    'master_audience': HUMAN / 'Master Audience Codebook - Final.xlsx',
    'inter_content': HUMAN / 'Intercoder Reliability Codebook - Content.xlsx',
    'inter_audience': HUMAN / 'Intercoder Reliability Codebook - Audience.xlsx',
    'llm_content': LLM / 'LLM Coding - Content Analysis.xlsx',
    'llm_audience': LLM / 'LLM Coding - Audience Analysis.xlsx',
}

for k,v in PATHS.items():
    if not v.exists():
        raise FileNotFoundError(f'Missing {k}: {v}')
print('All files found.')

In [ ]:
COUNTRIES = ['Nigeria', 'Kenya']

OPEN_TEXT = {
    'content': {'Context', 'Content Text / Description', 'Q1a', 'Q18a'},
    'audience': {'Context', 'Comment Text', 'Q7', 'Q21a', 'Q21c', 'Q21e', 'Q21g'},
}

MULTI = {
    'content': {'Q2','Q3','Q8','Q9','Q10','Q11','Q12','Q13','Q14','Q18a'},
    'audience': {'Q10'},
}

ID_COL = {'content':'Content ID','audience':'Comment ID'}

Q_RE = re.compile(r'^(Q\d+[a-z]?)', re.I)

def norm(x):
    if pd.isna(x):
        return None
    if isinstance(x,str):
        y=' '.join(x.replace(' ',' ').replace('​','').strip().split())
        return y if y else None
    return str(x).strip()

def key(x):
    y=norm(x)
    if y is None:
        return None
    y=y.lower().replace('&','and')
    y=re.sub(r'[^a-z0-9]+','',y)
    return y

def split_set(v):
    s=norm(v)
    if not s:
        return set()
    return {norm(t) for t in s.split(',') if norm(t)}

def jaccard(a,b):
    if not a and not b:
        return 1.0
    u=a|b
    if not u:
        return 1.0
    return len(a&b)/len(u)

def cohen_kappa(y_true, y_pred):
    if not y_true:
        return np.nan
    labels=sorted(set(y_true)|set(y_pred))
    if len(labels)<=1:
        return 1.0
    idx={l:i for i,l in enumerate(labels)}
    n=len(labels)
    mat=np.zeros((n,n),dtype=float)
    for t,p in zip(y_true,y_pred):
        mat[idx[t],idx[p]]+=1
    N=mat.sum()
    po=np.trace(mat)/N
    row=mat.sum(axis=1)/N
    col=mat.sum(axis=0)/N
    pe=float((row*col).sum())
    if np.isclose(1-pe,0):
        return np.nan
    return (po-pe)/(1-pe)

def sheet_df(path,sheet):
    df=pd.read_excel(path,sheet_name=sheet)
    for c in df.columns:
        if df[c].dtype==object:
            df[c]=df[c].map(norm)
    return df

def llm_q_map(df):
    out={}
    for c in df.columns:
        m=Q_RE.match(str(c))
        if m:
            q=m.group(1).upper()
            out[q]=c
    return out

In [ ]:
# Load workbooks
master_content = pd.concat([sheet_df(PATHS['master_content'], c) for c in COUNTRIES], ignore_index=True)
master_audience = pd.concat([sheet_df(PATHS['master_audience'], c) for c in COUNTRIES], ignore_index=True)
inter_content = pd.concat([sheet_df(PATHS['inter_content'], c) for c in COUNTRIES], ignore_index=True)
inter_audience = pd.concat([sheet_df(PATHS['inter_audience'], c) for c in COUNTRIES], ignore_index=True)

llm_content_ng = sheet_df(PATHS['llm_content'], 'Nigeria - LLM Coding')
llm_content_ke = sheet_df(PATHS['llm_content'], 'Kenya - LLM Coding')
llm_aud_ng = sheet_df(PATHS['llm_audience'], 'Nigeria - LLM Coding')
llm_aud_ke = sheet_df(PATHS['llm_audience'], 'Kenya - LLM Coding')

llm_content = pd.concat([llm_content_ng.assign(Country='Nigeria'), llm_content_ke.assign(Country='Kenya')], ignore_index=True)
llm_audience = pd.concat([llm_aud_ng.assign(Country='Nigeria'), llm_aud_ke.assign(Country='Kenya')], ignore_index=True)

print('master_content', master_content.shape)
print('master_audience', master_audience.shape)
print('inter_content', inter_content.shape)
print('inter_audience', inter_audience.shape)
print('llm_content', llm_content.shape)
print('llm_audience', llm_audience.shape)

In [ ]:
def scored_fields_human(df, track):
    cols=[]
    for c in df.columns:
        if not str(c).startswith('Q'):
            continue
        if c in OPEN_TEXT[track]:
            continue
        cols.append(c)
    return cols

def compute_human_human(inter_df, track, country):
    fields=scored_fields_human(inter_df, track)
    idcol=ID_COL[track]
    sub=inter_df[inter_df['Country']==country].copy()

    # QA checks
    assert len(sub)==60, f'{track}-{country}: expected 60 rows, got {len(sub)}'
    oc=sub.groupby('Overlap_Type')[idcol].count().to_dict()
    assert oc.get('pairwise',0)==30 and oc.get('all-coder',0)==30, f'Bad overlap composition {track}-{country}: {oc}'

    exact=[]
    overlap=[]
    by_field_pairs={f:[] for f in fields}
    alpha_by_field={}

    coders=sorted(sub['Coder'].dropna().unique().tolist())
    items=sorted(sub[idcol].dropna().unique().tolist())

    grouped=sub.groupby(idcol, dropna=False)
    for f in fields:
        for _,g in grouped:
            rows=g[['Coder',f]].dropna(subset=['Coder'])
            if len(rows)<2:
                continue
            for (_,r1),(_,r2) in itertools.combinations(rows.iterrows(),2):
                v1,v2=norm(r1[f]),norm(r2[f])
                if v1 is None or v2 is None:
                    continue
                if f in MULTI[track]:
                    s1,s2=split_set(v1),split_set(v2)
                    exact.append(1.0 if s1==s2 else 0.0)
                    overlap.append(jaccard(s1,s2))
                    by_field_pairs[f].append((' | '.join(sorted(s1)), ' | '.join(sorted(s2))))
                else:
                    exact.append(1.0 if key(v1)==key(v2) else 0.0)
                    overlap.append(1.0 if key(v1)==key(v2) else 0.0)
                    by_field_pairs[f].append((key(v1),key(v2)))

        mat=np.full((len(coders),len(items)), np.nan, dtype=float)
        ci={c:i for i,c in enumerate(coders)}
        ii={it:j for j,it in enumerate(items)}
        cat_to_int={}
        nxt=0
        for _,r in sub[[idcol,'Coder',f]].iterrows():
            if r['Coder'] not in ci or r[idcol] not in ii:
                continue
            v=norm(r[f])
            if v is None:
                continue
            if f in MULTI[track]:
                s=split_set(v)
                if not s:
                    continue
                v=' | '.join(sorted(s))
            k=key(v)
            if k not in cat_to_int:
                cat_to_int[k]=nxt
                nxt+=1
            mat[ci[r['Coder']], ii[r[idcol]]] = float(cat_to_int[k])
        try:
            alpha_by_field[f]=krippendorff.alpha(reliability_data=mat, level_of_measurement='nominal')
        except Exception:
            alpha_by_field[f]=np.nan

    kappas=[]
    for f,pairs in by_field_pairs.items():
        if not pairs:
            continue
        y1=[a for a,_ in pairs]
        y2=[b for _,b in pairs]
        k=cohen_kappa(y1,y2)
        if not pd.isna(k):
            kappas.append(k)

    alphas=[v for v in alpha_by_field.values() if not pd.isna(v)]

    summ={
        'metric_group':'human_human',
        'track':track,
        'country':country,
        'exact_agreement_pct':round(np.mean(exact)*100,1) if exact else np.nan,
        'set_overlap_pct':round(np.mean(overlap)*100,1) if overlap else np.nan,
        'kappa_mean':round(float(np.mean(kappas)),3) if kappas else np.nan,
        'alpha_mean':round(float(np.mean(alphas)),3) if alphas else np.nan,
        'alpha_min':round(float(np.min(alphas)),3) if alphas else np.nan,
        'alpha_max':round(float(np.max(alphas)),3) if alphas else np.nan,
        'alpha_fields_n':len(alphas),
    }
    fdf=pd.DataFrame({'track':track,'country':country,'field':list(alpha_by_field.keys()),'kripp_alpha':[alpha_by_field[k] for k in alpha_by_field]})
    return summ,fdf

In [ ]:
def compute_human_llm(master_df, llm_df, track, country):
    idcol=ID_COL[track]
    fields=scored_fields_human(master_df, track)

    h=master_df[master_df['Country']==country].copy()
    l=llm_df[llm_df['Country']==country].copy()

    lmap=llm_q_map(l)
    common=[f for f in fields if f in lmap]

    h[idcol]=h[idcol].map(norm)
    l[idcol]=l[idcol].map(norm)

    h_by={r[idcol]:r for _,r in h.iterrows() if norm(r[idcol])}
    l_by={r[idcol]:r for _,r in l.iterrows() if norm(r[idcol])}

    shared=sorted(set(h_by.keys()) & set(l_by.keys()))

    exact=[]
    overlap=[]
    kappas=[]

    for f in common:
        pairs=[]
        for _id in shared:
            hv=norm(h_by[_id].get(f))
            lv=norm(l_by[_id].get(lmap[f]))
            if hv is None or lv is None:
                continue
            if f in MULTI[track]:
                s1,s2=split_set(hv),split_set(lv)
                exact.append(1.0 if s1==s2 else 0.0)
                overlap.append(jaccard(s1,s2))
                pairs.append((' | '.join(sorted(s1)), ' | '.join(sorted(s2))))
            else:
                exact.append(1.0 if key(hv)==key(lv) else 0.0)
                overlap.append(1.0 if key(hv)==key(lv) else 0.0)
                pairs.append((key(hv), key(lv)))
        if pairs:
            k=cohen_kappa([a for a,_ in pairs],[b for _,b in pairs])
            if not pd.isna(k):
                kappas.append(k)

    return {
        'metric_group':'human_llm',
        'track':track,
        'country':country,
        'shared_ids':len(shared),
        'exact_agreement_pct':round(np.mean(exact)*100,1) if exact else np.nan,
        'set_overlap_pct':round(np.mean(overlap)*100,1) if overlap else np.nan,
        'kappa_mean':round(float(np.mean(kappas)),3) if kappas else np.nan,
    }

In [ ]:
summaries=[]
alpha_tables=[]

for track, inter_df in [('content', inter_content), ('audience', inter_audience)]:
    for country in COUNTRIES:
        s, a = compute_human_human(inter_df, track, country)
        summaries.append(s)
        alpha_tables.append(a)

for track, master_df, llm_df in [
    ('content', master_content, llm_content),
    ('audience', master_audience, llm_audience),
]:
    for country in COUNTRIES:
        summaries.append(compute_human_llm(master_df, llm_df, track, country))

summary_df=pd.DataFrame(summaries)
alpha_df=pd.concat(alpha_tables, ignore_index=True)
summary_df

In [ ]:
# Quick comparison table for report-facing numbers
pivot = summary_df.pivot_table(index=['metric_group','track'], columns='country', values=['exact_agreement_pct','set_overlap_pct','alpha_mean'], aggfunc='first')
pivot

In [ ]:

summary_xlsx = OUT / 'Research Grade Metrics Summary.xlsx'
alpha_xlsx = OUT / 'Research Grade Alpha By Field.xlsx'

with pd.ExcelWriter(summary_xlsx, engine='openpyxl') as w:
    summary_df.to_excel(w, index=False, sheet_name='Summary')

with pd.ExcelWriter(alpha_xlsx, engine='openpyxl') as w:
    alpha_df.to_excel(w, index=False, sheet_name='Alpha By Field')

print('Wrote', summary_xlsx)
print('Wrote', alpha_xlsx)

# High-agreement alpha fields >= 0.60
alpha_df[alpha_df['kripp_alpha'] >= 0.60].sort_values(['track','country','kripp_alpha'], ascending=[True,True,False])


## Interpretation note
Low overall alpha can occur in complex interpretive tasks with many categories. Surface-marker fields often yield higher agreement.